# Phase 1.4 — Understanding Sleep-EDF

I verify EEG channels, annotation conversions, dataset shape, subject grouping, and provenance.

> **Research integrity:** Figures labeled synthetic are not clinical EEG observations. Run code locally in VS Code using the project `.venv` interpreter.

## 1. Source and metadata

Sleep-EDF Expanded is hosted by PhysioNet. The project downloads its sleep-cassette subset using MNE. I must cite the dataset and preserve its original annotation rules.

In [ ]:
from eeg_sleep.constants import STAGES, EEG_PICKS, ANNOTATION_TO_STAGE
print("EEG derivations:", EEG_PICKS)
print("Mapped stages:", STAGES)
for source, target in ANNOTATION_TO_STAGE.items(): print(source, "→", target)

## 2. Filename grouping

Different nights from a single subject must share the **same** subject key during train/validation/test splitting.

In [ ]:
from eeg_sleep.dataset import subject_id_from_filename
for name in ["SC4001E0-PSG.edf", "SC4002E0-PSG.edf", "SC4011E0-PSG.edf"]:
    print(name, "=>", subject_id_from_filename(name))

## 3. Inspect a local dataset without downloading anything

This block generates a clearly labeled *synthetic* array only if no processed dataset exists. For real data use the explicit `fetch` then `prepare` CLI commands.

In [ ]:
from pathlib import Path
import numpy as np
from eeg_sleep.synthetic import make_demo_dataset
from eeg_sleep.dataset import load_arrays
path=Path("data/processed/synthetic_demo.npz")
if not path.exists(): make_demo_dataset(path, n_subjects=10, repeats_per_stage=1)
X,y,subjects,ids,fs=load_arrays(path)
print(f"Epochs={len(y)}, shape={X.shape}, fs={fs} Hz")
print("Subjects:", len(set(subjects)))
print("Samples per class:", dict(zip(STAGES,np.bincount(y,minlength=5))))

## 4. Real PhysioNet recordings (optional, requires external download)

In your VS Code Terminal run:

```bash
python scripts/run_pipeline.py fetch --subjects 0 1 --recordings 1
python scripts/run_pipeline.py prepare
```

Then inspect `data/processed/sleep_edf.npz` using `load_arrays`. Do not commit data files to Git.

In [ ]:
from pathlib import Path
from eeg_sleep.dataset import load_arrays
real_path=Path("data/processed/sleep_edf.npz")
if real_path.exists():
    real_X,real_y,real_groups,real_ids,real_fs=load_arrays(real_path)
    print("Real dataset shape:",real_X.shape,"Subjects:",len(set(real_groups)))
else:
    print("No real EDF data processed yet; this is expected before the download phase.")

## Exercises

1. Why do the two nights of SC400 share one group ID? 2. Why are original stages 3 and 4 merged? 3. Why is that not equivalent to AASM rescoring? 4. What is lost when EOG and EMG are ignored?